# Notebook 01 — Auditoria dos Dados

Auditoria exploratória da base de anúncios de imóveis de São Paulo (abril/2019), feita **antes** de qualquer modelagem. O objetivo desta etapa é entender a base, identificar e tratar problemas de qualidade de dado, e registrar decisões fundamentadas que vão sustentar todo o resto do projeto.

> **Princípio que guia esta auditoria:** resumo estatístico não substitui a inspeção das linhas. Vários problemas desta base só apareceram quando olhamos os registros individuais — não no `.info()` nem no `.describe()`.

## 1. Carregamento e inspeção inicial

Carregamos o CSV em um DataFrame e rodamos as três verificações de primeiro contato: dimensões (`shape`), aparência (`head`) e saúde (`info`) — esta última mostra o tipo de cada coluna e quantos valores não-nulos existem.

In [14]:
import pandas as pd

df = pd.read_csv("data/sao-paulo-properties-april-2019.csv")

print("Dimensões:", df.shape)

Dimensões: (13640, 16)


In [15]:
df.head()

,Price,Condo,Size,Rooms,Toilets,Suites,Parking,Elevator,Furnished,Swimming Pool,New,District,Negotiation Type,Property Type,Latitude,Longitude
0,930,220,47,2,2,1,1,0,0,0,0,Artur Alvim/São Paulo,rent,apartment,-23.543138,-46.479486
1,1000,148,45,2,2,1,1,0,0,0,0,Artur Alvim/São Paulo,rent,apartment,-23.550239,-46.480718
2,1000,100,48,2,2,1,1,0,0,0,0,Artur Alvim/São Paulo,rent,apartment,-23.542818,-46.485665
3,1000,200,48,2,2,1,1,0,0,0,0,Artur Alvim/São Paulo,rent,apartment,-23.547171,-46.483014
4,1300,410,55,2,2,1,1,1,0,0,0,Artur Alvim/São Paulo,rent,apartment,-23.525025,-46.482436


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13640 entries, 0 to 13639
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             13640 non-null  int64  
 1   Condo             13640 non-null  int64  
 2   Size              13640 non-null  int64  
 3   Rooms             13640 non-null  int64  
 4   Toilets           13640 non-null  int64  
 5   Suites            13640 non-null  int64  
 6   Parking           13640 non-null  int64  
 7   Elevator          13640 non-null  int64  
 8   Furnished         13640 non-null  int64  
 9   Swimming Pool     13640 non-null  int64  
 10  New               13640 non-null  int64  
 11  District          13640 non-null  str    
 12  Negotiation Type  13640 non-null  str    
 13  Property Type     13640 non-null  str    
 14  Latitude          13640 non-null  float64
 15  Longitude         13640 non-null  float64
dtypes: float64(2), int64(11), str(3)
memory usage: 1.7 

A base tem **13.640 imóveis e 16 colunas, sem nenhum valor nulo** (todas as colunas têm 13.640 registros não-nulos). Os tipos vieram coerentes: números como `int64`/`float64` e textos (`District`, `Negotiation Type`, `Property Type`) como `str`.

## 2. Separação por tipo de negociação

A coluna `Price` carrega duas grandezas muito diferentes: para imóveis de venda é o **valor de compra** (centenas de milhares de reais), e para locação é o **aluguel mensal** (milhares). Misturá-las contamina qualquer estatística ou modelo — uma "média" entre um aluguel de R$ 1.000 e uma venda de R$ 800.000 não significa nada.

Por isso, antes de qualquer análise, verificamos a composição e separamos a base em dois DataFrames, um por finalidade. Essa decisão se mantém até o fim do projeto: **dois modelos independentes**, porque os preços implícitos dos atributos são estruturalmente diferentes entre comprar e alugar.

In [17]:
df["Negotiation Type"].value_counts()

Negotiation Type
rent    7228
sale    6412
Name: count, dtype: int64

Agora vamos criar dois DataFrame novos, cada um vai ter imóveis de somente uma finalidade

In [18]:
df_venda = df[df["Negotiation Type"] == "sale"].copy()
df_aluguel = df[df["Negotiation Type"] == "rent"].copy()

print("Venda:", df_venda.shape)
print("Aluguel:", df_aluguel.shape)

Venda: (6412, 16)
Aluguel: (7228, 16)


A base tem **6.412 imóveis de venda e 7.228 de locação** — volumes saudáveis e equilibrados, o que viabiliza treinar um modelo independente para cada finalidade sem que nenhum fique com dados de menos.

*Nota técnica:* `df["Negotiation Type"] == "sale"` cria uma **máscara booleana** (uma série de True/False, uma por linha) e, passada de volta ao `df[...]`, retorna só as linhas marcadas — é o equivalente ao `WHERE` do SQL. O `.copy()` garante que cada novo DataFrame seja independente do original, e não apenas uma "vista" dele.

## 3. Auditoria dos preços

Com as grandezas separadas, analisamos a estatística descritiva do preço em cada finalidade, procurando três coisas: a **escala** (faz sentido?), os **extremos** (há valores impossíveis?) e a **distância entre média e mediana** (indício de assimetria na distribuição).

In [19]:
print("=== Preço de Venda ===")
print(df_venda["Price"].describe().map('{:.2f}'.format))
print("\n=== Preço de Aluguel ===")
print(df_aluguel["Price"].describe().map('{:.2f}'.format))

=== Preço de Venda ===
count        6412.00
mean       608624.14
std        740451.55
min         42000.00
25%        250000.00
50%        380000.00
75%        679000.00
max      10000000.00
Name: Price, dtype: str

=== Preço de Aluguel ===
count     7228.00
mean      3077.67
std       3522.83
min        480.00
25%       1350.00
50%       2000.00
75%       3300.00
max      50000.00
Name: Price, dtype: str


A leitura confirma dois pontos:

- **Escala coerente:** venda gira em centenas de milhares (mediana R$ 380 mil) e locação em milhares (mediana R$ 2.000) — a separação fez sentido.
- **Cauda à direita:** em ambas, a **média é bem maior que a mediana** (venda: R$ 608 mil vs R$ 380 mil; locação: R$ 3.078 vs R$ 2.000). É a assinatura de uma distribuição assimétrica, com poucos imóveis caros puxando a média para cima.

Os extremos da venda chamam atenção e merecem investigação antes de qualquer conclusão: um mínimo de R$ 42 mil (baixo para a capital) e um máximo de R$ 10 milhões "redondo" (suspeito de teto de cadastro). Vamos olhar as **linhas inteiras** dos dez mais baratos e dos dez mais caros — porque só o preço não diz se é erro ou mercado real; é preciso ver os atributos junto.

In [20]:
df_venda.sort_values("Price").head(10)

,Price,Condo,Size,Rooms,Toilets,Suites,Parking,Elevator,Furnished,Swimming Pool,New,District,Negotiation Type,Property Type,Latitude,Longitude
8487,42000,300,48,1,1,0,1,0,1,0,0,República/São Paulo,sale,apartment,0.000000,0.000000
6421,45000,225,40,2,2,1,1,0,1,0,0,Lajeado/São Paulo,sale,apartment,-23.544439,-46.405896
12714,47481,0,47,2,1,0,0,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,0.000000,0.000000
11449,60000,60,42,2,2,1,1,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.590204,-46.398523
6201,65000,110,48,2,2,1,1,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.590323,-46.399616
11450,66000,110,45,2,2,1,1,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.595000,-46.408663
11377,68000,0,90,3,2,1,2,1,0,1,0,Aricanduva/São Paulo,sale,apartment,-46.539384,-23.535175
6199,70000,0,48,2,2,1,1,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.593677,-46.398922
6200,72000,0,50,2,2,1,1,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.592982,-46.406661
12719,74481,0,44,2,1,0,0,0,0,0,0,Cidade Tiradentes/São Paulo,sale,apartment,-23.600647,-46.401390


In [21]:
df_venda.sort_values("Price", ascending=False).head(10)

,Price,Condo,Size,Rooms,Toilets,Suites,Parking,Elevator,Furnished,Swimming Pool,New,District,Negotiation Type,Property Type,Latitude,Longitude
6282,10000000,0,343,4,7,4,5,0,0,0,0,Iguatemi/São Paulo,sale,apartment,-23.585487,-46.681676
6287,9979947,0,343,4,6,4,5,1,0,1,0,Iguatemi/São Paulo,sale,apartment,-23.585487,-46.681676
7249,8500000,7200,420,4,6,4,4,1,0,1,0,Jardim Paulista/São Paulo,sale,apartment,-23.564044,-46.660862
5008,8039200,0,278,4,7,4,4,1,1,1,0,Vila Olimpia/São Paulo,sale,apartment,-23.596469,-46.680587
7152,8000000,0,269,4,5,4,4,1,0,1,0,Itaim Bibi/São Paulo,sale,apartment,-23.588294,-46.680764
5006,8000000,0,278,4,5,3,5,1,0,1,0,Vila Olimpia/São Paulo,sale,apartment,-23.596469,-46.680587
6286,7765616,0,279,4,6,4,4,1,0,1,0,Iguatemi/São Paulo,sale,apartment,-23.585487,-46.681676
8935,7559420,0,377,3,4,3,4,0,0,1,0,Alto de Pinheiros/São Paulo,sale,apartment,-23.549506,-46.717449
7623,7559420,0,377,3,4,3,4,1,0,1,0,Alto de Pinheiros/São Paulo,sale,apartment,-23.549505,-46.717448
8949,7521000,2500,377,6,7,6,4,0,0,1,0,Alto de Pinheiros/São Paulo,sale,apartment,-23.549506,-46.717449


### O que os extremos nos contam

Investigando as duas pontas, concluímos que **os preços estão saudáveis em toda a faixa** — não há outliers de preço a remover:

- **Os mais baratos são reais.** Quase todos abaixo de R$ 75 mil estão em Cidade Tiradentes e Lajeado, no extremo leste — a região mais barata da cidade. Apartamentos de 40–50 m² por R$ 60–72 mil ali, em 2019, são plausíveis. Aquele mínimo de R$ 42 mil que parecia erro é, na verdade, mercado de periferia.
- **Os mais caros também são reais.** Os de R$ 7–10 milhões estão em Iguatemi, Jardim Paulista, Vila Olímpia, Itaim Bibi e Alto de Pinheiros (bairros nobres), com 270–420 m² e 4 suítes. É luxo legítimo. O valor de R$ 10 milhões "redondo" tem um vizinho de R$ 9.979.947 com a mesma metragem no mesmo bairro — ou seja, é preço de mercado, não teto de cadastro.

> **Decisão (preço):** nenhuma remoção por preço. A assimetria à direita (média > mediana) confirma que vamos modelar o **log do preço** mais adiante.

Porém, a inspeção das linhas revelou um problema que **não é de preço**: vários registros têm o valor **zero** onde zero não deveria existir — coordenadas em
(0, 0), e também condomínio, e potencialmente outros campos. Isso sugere que o zero, nesta base, às vezes não é um fato, e sim a marca de uma informação que faltava no
anúncio. Antes de seguir, precisamos entender esse fenômeno de forma global, coluna a coluna — é o que a próxima seção faz.

## 4. Auditoria dos zeros — quando o zero é um missing disfarçado

O `info()` mostrou que não há valores nulos, mas isso é enganoso: nesta base, a
ausência de informação costuma ser preenchida com **zero** em vez de vazio. Um zero
desses não é um fato sobre o imóvel — é um buraco de cadastro que o `.info()` não
enxerga, porque tecnicamente "tem valor".

O ponto central é que **a semântica do zero depende da coluna**. O mesmo número `0`
significa coisas opostas conforme onde aparece, e podemos agrupar as colunas em três
categorias:

- **Zero impossível (sempre missing disfarçado):** colunas onde zero não pode
  existir na realidade — `Price` (imóvel de graça?), `Size` (apartamento sem área?),
  `Rooms` e `Toilets` (não existe apartamento sem banheiro).
  Aqui, todo zero é ausência de informação.
- **Zero ambíguo (depende — o mais perigoso):** `Condo` é o caso típico. Um
  condomínio zerado *pode* ser real (imóvel sem taxa) ou *pode* ser "o anúncio não
  informou". Os dois cenários convivem na mesma coluna e exigem julgamento.
- **Zero legítimo (é um valor real, não mexer):** as colunas binárias `Elevator`,
  `Furnished`, `Swimming Pool`, `New`, onde zero significa "não tem" — informação
  verdadeira e essencial. `Suites` e `Parking` também entram aqui (zero suíte ou
  zero vaga são reais).

> **Por que isso importa:** uma regra global ingênua do tipo "todo zero é missing"
> destruiria as colunas binárias, transformando todo apartamento sem piscina num
> dado faltante. Por isso não tratamos o zero com machado — primeiro **medimos**
> quanto zero existe em cada coluna, e só depois decidimos o tratamento de cada uma
> conforme sua categoria.

Medir antes de agir — o mesmo princípio que aplicamos aos outliers de preço.

In [22]:
def diagnostico_zeros(dados, nome):
    zeros = (dados == 0).sum()                      # conta zeros por coluna
    pct = (zeros / len(dados) * 100).round(1)       # converte em percentual
    resultado = pd.DataFrame({"qtd_zeros": zeros, "pct_zeros": pct})
    # mostra só as colunas que TÊM algum zero, da pior para a melhor
    resultado = resultado[resultado["qtd_zeros"] > 0].sort_values("pct_zeros", ascending=False)
    print(f"=== {nome} ({len(dados)} imóveis) ===")
    print(resultado)
    print()

diagnostico_zeros(df_venda, "VENDA")
diagnostico_zeros(df_aluguel, "ALUGUEL")

=== VENDA (6412 imóveis) ===
               qtd_zeros  pct_zeros
New                 6205       96.8
Furnished           5660       88.3
Elevator            3748       58.5
Swimming Pool       2953       46.1
Suites              1570       24.5
Condo               1337       20.9
Latitude             398        6.2
Longitude            398        6.2
Parking              276        4.3

=== ALUGUEL (7228 imóveis) ===
               qtd_zeros  pct_zeros
New                 7222       99.9
Furnished           5978       82.7
Elevator            5061       70.0
Swimming Pool       3701       51.2
Suites              1715       23.7
Condo                640        8.9
Latitude             483        6.7
Longitude            483        6.7
Parking              294        4.1



### Diagnóstico: o que os zeros revelaram

O mapeamento trouxe três conclusões — uma tranquilizadora, uma que confirma a
suspeita do condomínio, e uma nuance de modelagem.

**1. As colunas onde o zero seria impossível estão limpas.** `Price`, `Size`,
`Rooms` e `Toilets` não aparecem na lista — têm zero ocorrências de zero. Não
existe, nesta base, apartamento sem área, sem quarto, sem banheiro ou de graça. A
preocupação que motivou esta auditoria não se materializou nessas colunas; o único
"zero impossível" que de fato ocorre é o de coordenadas.

**2. O `Condo` zerado é, quase certamente, informação ausente — e os dados o
provam.** São 20,9% de zeros na venda contra apenas 8,9% no aluguel. Como a base é
só de apartamentos (todos com taxa de condomínio real), e como é o mesmo estoque de
imóveis anunciado nas duas pontas, uma diferença dessas só se explica por
comportamento de anúncio: no aluguel o condomínio compõe o custo mensal e é quase
sempre informado; na venda é secundário e frequentemente omitido. Portanto,
tratamos `Condo = 0` como **missing**, não como isenção — deixá-lo como zero faria
o modelo enxergar esses imóveis como "sem custo de condomínio", enviesando o preço
para baixo.

**3. As binárias com zero altíssimo são legítimas, mas algumas são features
fracas.** `New` (96,8% na venda, 99,9% no aluguel) e `Furnished` (≈85%) têm
variância baixíssima — são quase constantes. O zero é real ("não é lançamento", "não
mobiliado"), então não há o que limpar, mas `New` no aluguel é praticamente inútil
como preditor (quase não varia).

### Tabela de decisão por coluna

| Coluna(s) | Categoria do zero | O que o zero significa | Tratamento |
|---|---|---|---|
| `Price`, `Size`, `Rooms`, `Toilets` | — (sem zeros) | n/a | Nada a fazer — limpas |
| `Latitude`, `Longitude` | Impossível → missing | Sem geolocalização | Tratar (881 registros) — ver sub-análise abaixo |
| `Condo` | Ambíguo → missing | Anúncio não informou a taxa | Tratar como missing na fase de tratamento (flag + imputação); **não** deixar como 0 |
| `New` | Legítimo | Não é lançamento | Manter (variância baixa — feature fraca) |
| `Furnished` | Legítimo | Não mobiliado | Manter (variância baixa) |
| `Elevator` | Legítimo | Sem elevador | Manter |
| `Swimming Pool` | Legítimo | Sem piscina | Manter |
| `Suites` | Legítimo | Sem suíte | Manter |
| `Parking` | Legítimo | Sem vaga | Manter |  

> **Decisão (zeros):** as colunas binárias e de contagem (`Suites`, `Parking`)
> ficam como estão — zero é informação real. `Condo = 0` passa a ser tratado como
> dado ausente na fase de tratamento (não como valor). As coordenadas (0, 0) entram
> na sub-análise de geolocalização a seguir, por exigirem um tratamento próprio

## 5. Auditoria da geolocalização

A tabela de decisão da seção anterior deixou as coordenadas pendentes de tratamento
próprio, e por um bom motivo: a geolocalização tem **dois** defeitos distintos, e
apenas um deles aparece no diagnóstico de zeros.

- **Coordenadas (0, 0) — já quantificadas:** 881 imóveis (398 de venda + 483 de
  aluguel) com latitude e/ou longitude zerada. É "zero impossível" — um imóvel em
  São Paulo não fica na coordenada (0, 0), no meio do Atlântico. Trata-se de
  geolocalização ausente.
- **Coordenadas invertidas — ainda não investigadas:** ao inspecionar linhas na
  auditoria de preços, notamos registros com latitude e longitude **trocadas de
  campo**. Esses *não* aparecem no diagnóstico de zeros, porque os valores não são
  zero — são válidos, só estão na coluna errada. Exigem uma caça própria.

Para encontrá-los, usamos o conhecimento do território: **São Paulo está em latitude
≈ -23 e longitude ≈ -46**. Logo, qualquer registro cuja `Latitude` esteja próxima de
-46 (abaixo de -40) só pode estar com os dois valores trocados. É um filtro simples
que usa a geografia da cidade para isolar o erro.

Esta seção faz três coisas, nesta ordem: (1) caça e **exibe** os registros
invertidos — o defeito que ainda não havíamos mostrado; (2) verifica se os 881
zerados estão espalhados ou concentrados, para decidir se a remoção é segura ou
enviesada; e só então definiremos o tratamento de cada caso.

In [23]:
# Recriamos os dois subconjuntos de coordenada defeituosa para tratá-los.

# (1) Coordenadas zeradas
zerados = df[(df["Latitude"] == 0) | (df["Longitude"] == 0)]

# (2) Coordenadas invertidas, latitude no intervalo da longitude (< -40)
invertidos = df[df["Latitude"] < -40]

print("Coordenadas (0,0):     ", len(zerados))
print("Coordenadas invertidas:", len(invertidos))

Coordenadas (0,0):      881
Coordenadas invertidas: 17


In [24]:
# Os 17 registros invertidos, de perto.
invertidos[["District", "Latitude", "Longitude", "Price", "Negotiation Type"]]

,District,Latitude,Longitude,Price,Negotiation Type
1113,Jabaquara/São Paulo,-46.648904,-23.652027,1300,rent
1211,Moema/São Paulo,-46.655399,-23.607013,2200,rent
1583,Alto de Pinheiros/São Paulo,-46.715115,-23.540783,2600,rent
1792,Jaguaré/São Paulo,-46.749039,-23.545329,1400,rent
1937,Perdizes/São Paulo,-46.678478,-23.534683,4200,rent
1962,Pinheiros/São Paulo,-46.700223,-23.568745,4000,rent
5545,Consolação/São Paulo,-46.648555,-23.548484,380000,sale
6888,Cambuci/São Paulo,-46.626667,-23.577821,490000,sale
9773,Vila Curuçá/São Paulo,-46.428927,-23.517640,1100,rent
9811,Vila Prudente/São Paulo,-46.577355,-23.598180,1700,rent


In [25]:
# Os 881 zerados estão espalhados pela cidade ou concentrados em poucos bairros?
print("Por tipo de negociação:")
print(zerados["Negotiation Type"].value_counts())
print()
print("Bairros distintos afetados:", zerados["District"].nunique())
print()
print("Top 15 bairros com mais coordenadas zeradas:")
print(zerados["District"].value_counts().head(15))

Por tipo de negociação:
Negotiation Type
rent    483
sale    398
Name: count, dtype: int64

Bairros distintos afetados: 83

Top 15 bairros com mais coordenadas zeradas:
District
Artur Alvim/São Paulo      60
Casa Verde/São Paulo       28
São Mateus/São Paulo       24
Brás/São Paulo             23
Cidade Líder/São Paulo     23
Bela Vista/São Paulo       22
Consolação/São Paulo       22
Carrão/São Paulo           21
Santa Cecília/São Paulo    20
Santana/São Paulo          20
Liberdade/São Paulo        19
Penha/São Paulo            19
Cambuci/São Paulo          19
Tatuapé/São Paulo          18
Barra Funda/São Paulo      17
Name: count, dtype: int64


### Decisão sobre a geolocalização

As duas investigações resolveram os dois defeitos em direções opostas — um recupera,
o outro remove.

**Os 17 invertidos são recuperáveis (campo trocado).** Em todos, a `Latitude` está
em ≈ -46 e a `Longitude` em ≈ -23 — exatamente os intervalos um do outro. Ao trocar
os valores de volta, cada coordenada cai dentro do bairro indicado em `District`
(conferido em Alto de Pinheiros, Aricanduva, Barra Funda, entre outros). Não é
corrupção de dado, é inversão de campo no cadastro. **Tratamento: corrigir trocando
`Latitude` e `Longitude` — recuperamos os 17.**

**Os 881 zerados têm ausência difusa — remoção segura.** A pergunta crítica era se a
ausência seria sistemática (geraria viés de seleção) ou aleatória. Três evidências
indicam aleatoriedade:

- Espalhados por **83 bairros distintos** — não concentrados em uma região.
- Equilibrados entre finalidades (483 aluguel, 398 venda) — nenhum modelo sofre
  desproporcionalmente.
- Topo da lista geograficamente heterogêneo: Artur Alvim (leste), Casa Verde
  (norte), Bela Vista e Consolação (centro), Santana (norte) — perfis distintos. Um
  viés deixaria um padrão; aqui há variedade.

A causa provável é técnica (falha pontual de geocodificação na coleta), não um viés
contra algum tipo de imóvel. Como a base não tem endereço para recuperar a
coordenada exata, e como o centroide do bairro distorceria o *spatial lag*,
**tratamento: remover os 881 registros.** Perder 6,5% distribuídos uniformemente é
preferível a contaminar a feature espacial mais importante do projeto.

> **Ponto de atenção:** Artur Alvim concentra 60 zeros (o segundo colocado tem 28).
> Antes de modelar, confirmar que ainda restam imóveis com coordenada válida nesse
> bairro, para não deixar o modelo sem referência local ali.

> **Decisão (geolocalização):** corrigir os 17 invertidos por troca de campo;
> remover os 881 com coordenada (0, 0). Ambas as operações serão executadas na
> próxima seção, junto com o tratamento do `Condo = 0` definido anteriormente.

## 6. Tratamento dos dados

Encerrado o diagnóstico, esta seção **modifica** os dados pela primeira vez —
aplicando as decisões que documentamos nas auditorias anteriores. É a fronteira do
notebook: tudo antes foi investigação.

Uma regra nos guia: **nunca sobrescrever o DataFrame original.** Criamos uma
versão tratada (`df_tratado`) e trabalhamos sobre ela. Assim, se precisarmos revisar
qualquer decisão, o `df` original continua intacto e a auditoria permanece
reproduzível.

Três tratamentos serão aplicados, na ordem:

1. **Corrigir os 17 invertidos** — trocar `Latitude` e `Longitude` de volta
   (recupera 17 imóveis).
2. **Remover os 881 zerados** — registros com coordenada (0, 0), sem geolocalização
   recuperável.
3. **Marcar `Condo = 0` como missing** — substituir o zero por `NaN`, para que a
   imputação na fase de modelagem o trate como ausente, não como "sem custo".

A ordem entre os passos 1 e 2 é indiferente (são conjuntos disjuntos: um tem
latitude ≈ -46, o outro tem latitude = 0), mas corrigimos antes de remover por
clareza de leitura.

In [ ]:
df_tratado = df.copy()

# --- Subpasso 1: corrigir as 17 coordenadas invertidas ---
mask_invertidos = df_tratado["Latitude"] < -40

lat_antiga = df_tratado.loc[mask_invertidos, "Latitude"].copy()
df_tratado.loc[mask_invertidos, "Latitude"] = df_tratado.loc[mask_invertidos, "Longitude"]
df_tratado.loc[mask_invertidos, "Longitude"] = lat_antiga

# Conferência: nenhum registro deve mais ter latitude < -40
print("Invertidos restantes:", (df_tratado["Latitude"] < -40).sum())

Invertidos restantes: 0


**Subpasso 1 concluído.** A conferência retornou `0` invertidos restantes — a troca
de `Latitude` ↔ `Longitude` foi aplicada com sucesso aos 17 registros, que agora têm
coordenadas válidas dentro de São Paulo. Nenhum imóvel foi perdido: recuperamos os
17 em vez de descartá-los.

In [27]:
# --- Subpasso 2: remover os 881 registros com coordenada (0, 0) ---
linhas_antes = len(df_tratado)

df_tratado = df_tratado[(df_tratado["Latitude"] != 0) & (df_tratado["Longitude"] != 0)]

linhas_depois = len(df_tratado)
print("Linhas antes: ", linhas_antes)
print("Linhas depois:", linhas_depois)
print("Removidas:    ", linhas_antes - linhas_depois)

Linhas antes:  13640
Linhas depois: 12759
Removidas:     881


**Subpasso 2 concluído.** Removidas exatamente **881 linhas** (de 13.640 para
12.759), batendo com o total de coordenadas (0, 0) diagnosticado na auditoria de
geolocalização. A base tratada mantém 12.759 imóveis com coordenada válida.

In [28]:
# --- Subpasso 3: marcar Condo = 0 como dado ausente (NaN) ---
import numpy as np

# Quantos serão afetados? (conferência ANTES)
print("Condo = 0 antes:", (df_tratado["Condo"] == 0).sum())

# A troca: onde Condo for 0, vira NaN
df_tratado["Condo"] = df_tratado["Condo"].replace(0, np.nan)

# Conferência DEPOIS: não deve sobrar nenhum zero, e devem surgir NaNs
print("Condo = 0 depois:", (df_tratado["Condo"] == 0).sum())
print("Condo NaN depois:", df_tratado["Condo"].isna().sum())

Condo = 0 antes: 1887
Condo = 0 depois: 0
Condo NaN depois: 1887


**Subpasso 3 concluído.** Os **1.887** registros com `Condo = 0` foram convertidos em
`NaN` (missing), sem sobrar nenhum zero. A coluna agora distingue corretamente "custo
de condomínio ausente no anúncio" de um valor real — a imputação desse missing fica
para a fase de modelagem.

### Balanço da base tratada

Partimos de 13.640 registros brutos e chegamos a uma base tratada e auditada:

| Operação | Efeito |
|---|---|
| Correção de coordenadas invertidas | 17 imóveis recuperados (lat/lon trocadas) |
| Remoção de coordenadas (0, 0) | 881 imóveis removidos (sem geolocalização) |
| `Condo = 0` → `NaN` | 1.887 valores marcados como ausentes |
| **Base final** | **12.759 imóveis com coordenada válida** |

O `df` original permanece intocado; todo o tratamento vive em `df_tratado`. As
decisões de imputação (com o que preencher os `NaN` de `Condo`) e de transformação
(log do preço) pertencem à fase de modelagem, não à de limpeza — mantemos a
auditoria e o tratamento separados da modelagem por princípio de reprodutibilidade.

In [29]:
# Conferência final da base tratada
print("Shape final:", df_tratado.shape)
print()
df_tratado.info()

Shape final: (12759, 16)

<class 'pandas.DataFrame'>
Index: 12759 entries, 0 to 13639
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             12759 non-null  int64  
 1   Condo             10872 non-null  float64
 2   Size              12759 non-null  int64  
 3   Rooms             12759 non-null  int64  
 4   Toilets           12759 non-null  int64  
 5   Suites            12759 non-null  int64  
 6   Parking           12759 non-null  int64  
 7   Elevator          12759 non-null  int64  
 8   Furnished         12759 non-null  int64  
 9   Swimming Pool     12759 non-null  int64  
 10  New               12759 non-null  int64  
 11  District          12759 non-null  str    
 12  Negotiation Type  12759 non-null  str    
 13  Property Type     12759 non-null  str    
 14  Latitude          12759 non-null  float64
 15  Longitude         12759 non-null  float64
dtypes: float64(3), int64(10), str(

**Conferência final aprovada.** A base tratada tem `(12759, 16)`, com as colunas
coerentes: `Condo` exibe 10.872 non-nulls (12.759 − 1.887 = os NaN criados) e passou
a `float64`, enquanto as demais permanecem completas. O índice preserva os rótulos
originais (0 a 13639, com saltos onde houve remoção), mantendo a rastreabilidade dos
imóveis. A base está limpa, auditada e pronta para a próxima fase.